# Driver Drowsiness Detection — Model Comparison Experiment

## Objective
Compare **YOLO detectors** (v8, v9, v11, v12) and **CNN classifiers** (CustomCNN, ResNet18, ResNet50, VGG16, EfficientNet-B0, MobileNetV2) to find the optimal detection + classification combo for real-time drowsiness detection.

### Metrics
- **YOLO**: Inference speed (ms/frame), FPS, detection accuracy
- **CNN**: Training accuracy, validation accuracy, inference speed, model size

In [ ]:
import sys, time
from pathlib import Path
PROJECT_ROOT = str(Path.cwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

---
## Part 1: YOLO Face Detector Comparison
Benchmark YOLOv8n, YOLOv9c, YOLO11n, YOLO12n on synthetic frames.

In [ ]:
from ultralytics import YOLO

YOLO_MODELS = ['yolov8n.pt', 'yolov9c.pt', 'yolo11n.pt', 'yolo12n.pt']
N_FRAMES = 20
N_WARMUP = 3

# Generate test frames (random + gradient to simulate real scenes)
test_frames = [np.random.randint(50, 200, (480, 640, 3), dtype=np.uint8) for _ in range(N_FRAMES)]

yolo_results = []
for model_name in YOLO_MODELS:
    print(f'\nBenchmarking {model_name}...')
    try:
        model = YOLO(model_name)
        # Warmup
        for i in range(N_WARMUP):
            model.predict(test_frames[i], verbose=False)
        # Timed runs
        times = []
        total_dets = 0
        for frame in test_frames:
            t0 = time.perf_counter()
            results = model.predict(frame, verbose=False, conf=0.4)
            elapsed = (time.perf_counter() - t0) * 1000
            times.append(elapsed)
            total_dets += len(results[0].boxes) if results[0].boxes is not None else 0
        avg_ms = np.mean(times)
        yolo_results.append({
            'Model': model_name.replace('.pt',''),
            'Avg (ms)': round(avg_ms, 2),
            'Std (ms)': round(np.std(times), 2),
            'Min (ms)': round(np.min(times), 2),
            'Max (ms)': round(np.max(times), 2),
            'FPS': round(1000/avg_ms, 1),
            'Total Dets': total_dets,
        })
        print(f'  Avg: {avg_ms:.1f}ms | FPS: {1000/avg_ms:.1f} | Detections: {total_dets}')
    except Exception as e:
        print(f'  FAILED: {e}')
        yolo_results.append({'Model': model_name.replace('.pt',''), 'Avg (ms)': 'N/A', 'FPS': 'N/A'})

yolo_df = pd.DataFrame(yolo_results)
print('\n=== YOLO Detector Comparison ===')
print(yolo_df.to_string(index=False))

In [ ]:
# Plot YOLO speed comparison
valid = [r for r in yolo_results if isinstance(r.get('Avg (ms)'), (int, float))]
if valid:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    names = [r['Model'] for r in valid]
    speeds = [r['Avg (ms)'] for r in valid]
    fps_vals = [r['FPS'] for r in valid]
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    ax1.barh(names, speeds, color=colors[:len(names)])
    ax1.set_xlabel('Inference Time (ms)')
    ax1.set_title('YOLO Inference Speed (lower = better)')
    for i, v in enumerate(speeds): ax1.text(v+0.5, i, f'{v:.1f}ms', va='center')
    ax2.barh(names, fps_vals, color=colors[:len(names)])
    ax2.set_xlabel('Frames Per Second')
    ax2.set_title('YOLO FPS (higher = better)')
    for i, v in enumerate(fps_vals): ax2.text(v+0.5, i, f'{v:.1f}', va='center')
    plt.tight_layout()
    plt.savefig('models/results/yolo_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Part 2: CNN Classifier Comparison
Compare 6 architectures on parameter count, size, and inference speed.

In [ ]:
from src.classification.model_builder import SUPPORTED_MODELS, build_model, model_summary

# Model size comparison
cnn_info = []
for name in SUPPORTED_MODELS:
    info = model_summary(name)
    cnn_info.append(info)

cnn_df = pd.DataFrame(cnn_info)
print('=== CNN Architecture Summary ===')
print(cnn_df.to_string(index=False))

In [ ]:
# CNN inference speed benchmark
N_ITERS = 50
dummy_batch = torch.randn(1, 3, 224, 224, device=device)

cnn_speed = []
for name in SUPPORTED_MODELS:
    model = build_model(name, num_classes=2, pretrained=False).to(device).eval()
    # Warmup
    with torch.no_grad():
        for _ in range(5): model(dummy_batch)
    # Timed
    times = []
    with torch.no_grad():
        for _ in range(N_ITERS):
            if device.type == 'cuda': torch.cuda.synchronize()
            t0 = time.perf_counter()
            out = model(dummy_batch)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    avg = np.mean(times)
    info = model_summary(name)
    cnn_speed.append({
        'Model': name,
        'Params (M)': round(info['total_params']/1e6, 2),
        'Size (MB)': info['size_mb'],
        'Avg (ms)': round(avg, 2),
        'FPS': round(1000/avg, 1),
    })
    print(f'{name:<18} | {avg:.2f}ms | {1000/avg:.0f} FPS | {info["total_params"]/1e6:.2f}M params')

speed_df = pd.DataFrame(cnn_speed)
print('\n=== CNN Speed Comparison ===')
print(speed_df.to_string(index=False))

In [ ]:
# Plot CNN comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7','#DDA0DD']
names = [r['Model'] for r in cnn_speed]

# Params
params = [r['Params (M)'] for r in cnn_speed]
axes[0].barh(names, params, color=colors)
axes[0].set_xlabel('Parameters (Millions)')
axes[0].set_title('Model Size (params)')
for i,v in enumerate(params): axes[0].text(v+0.1, i, f'{v:.1f}M', va='center', fontsize=9)

# Speed
spd = [r['Avg (ms)'] for r in cnn_speed]
axes[1].barh(names, spd, color=colors)
axes[1].set_xlabel('Inference (ms)')
axes[1].set_title('Inference Speed (lower = better)')
for i,v in enumerate(spd): axes[1].text(v+0.1, i, f'{v:.1f}ms', va='center', fontsize=9)

# FPS
fps = [r['FPS'] for r in cnn_speed]
axes[2].barh(names, fps, color=colors)
axes[2].set_xlabel('FPS')
axes[2].set_title('Throughput (higher = better)')
for i,v in enumerate(fps): axes[2].text(v+1, i, f'{v:.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('models/results/cnn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 3: Training All CNN Models (on dataset)
Train each model and compare val accuracy. **Skip if no dataset available.**

In [ ]:
from src.classification.train import create_data_loaders, train_model

DATA_DIR = 'data/processed'
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-4

try:
    train_loader, val_loader, test_loader = create_data_loaders(
        data_dir=DATA_DIR, batch_size=BATCH_SIZE, img_size=224, num_workers=4
    )
    DATASET_AVAILABLE = True
    print('Dataset loaded successfully!')
except FileNotFoundError:
    DATASET_AVAILABLE = False
    print('Dataset not found. Skipping training cells.')
    print('Run: python data/scripts/prepare_dataset.py')

In [ ]:
if DATASET_AVAILABLE:
    training_results = {}
    for model_name in SUPPORTED_MODELS:
        print(f'\n{"="*60}')
        print(f'Training: {model_name}')
        print(f'{"="*60}')
        history = train_model(
            model_name=model_name,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=EPOCHS,
            lr=LR,
            device=device,
        )
        training_results[model_name] = history
    print('\nAll models trained!')
else:
    print('Skipped — no dataset.')

In [ ]:
if DATASET_AVAILABLE and training_results:
    # Training curves
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    for name, hist in training_results.items():
        epochs = range(1, len(hist['train_loss'])+1)
        axes[0,0].plot(epochs, hist['train_loss'], label=name)
        axes[0,1].plot(epochs, hist['train_acc'], label=name)
        axes[1,0].plot(epochs, hist['val_loss'], label=name)
        axes[1,1].plot(epochs, hist['val_acc'], label=name)
    axes[0,0].set_title('Train Loss'); axes[0,0].legend(fontsize=8)
    axes[0,1].set_title('Train Accuracy'); axes[0,1].legend(fontsize=8)
    axes[1,0].set_title('Val Loss'); axes[1,0].legend(fontsize=8)
    axes[1,1].set_title('Val Accuracy'); axes[1,1].legend(fontsize=8)
    for ax in axes.flat: ax.set_xlabel('Epoch'); ax.grid(True, alpha=0.3)
    plt.suptitle('CNN Training Comparison', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.savefig('models/results/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Summary table
    summary = []
    for name, hist in training_results.items():
        info = model_summary(name)
        summary.append({
            'Model': name,
            'Best Val Acc': f"{hist['best_val_acc']:.4f}",
            'Best Epoch': hist['best_epoch'],
            'Params (M)': round(info['total_params']/1e6, 2),
            'Time (s)': round(hist['training_time_s'], 1),
        })
    print('\n=== TRAINING RESULTS ===')
    print(pd.DataFrame(summary).to_string(index=False))

---
## Part 4: Combined Pipeline Speed (YOLO + CNN)
Estimate total pipeline latency for each YOLO + CNN combination.

In [ ]:
# Build combined speed table
valid_yolo = [r for r in yolo_results if isinstance(r.get('Avg (ms)'), (int, float))]
combo_data = []
for yr in valid_yolo:
    for cr in cnn_speed:
        total = yr['Avg (ms)'] + cr['Avg (ms)']
        combo_data.append({
            'YOLO': yr['Model'],
            'CNN': cr['Model'],
            'YOLO (ms)': yr['Avg (ms)'],
            'CNN (ms)': cr['Avg (ms)'],
            'Total (ms)': round(total, 2),
            'Pipeline FPS': round(1000/total, 1),
        })

combo_df = pd.DataFrame(combo_data)
combo_df = combo_df.sort_values('Total (ms)')
print('=== Top 10 Fastest YOLO + CNN Combos ===')
print(combo_df.head(10).to_string(index=False))
print(f'\nFastest combo: {combo_df.iloc[0]["YOLO"]} + {combo_df.iloc[0]["CNN"]} @ {combo_df.iloc[0]["Pipeline FPS"]} FPS')

In [ ]:
# Heatmap of pipeline speeds
pivot = combo_df.pivot(index='CNN', columns='YOLO', values='Pipeline FPS')
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.0f}', ha='center', va='center', fontsize=11, fontweight='bold')
ax.set_title('Pipeline FPS: YOLO Detector + CNN Classifier (higher = better)', fontsize=13)
ax.set_xlabel('YOLO Detector'); ax.set_ylabel('CNN Classifier')
plt.colorbar(im, label='FPS')
plt.tight_layout()
plt.savefig('models/results/pipeline_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5: EAR/MAR Sensitivity Analysis

In [ ]:
from src.utils.drowsiness_utils import compute_drowsiness_score, drowsiness_level

# Sweep EAR and CNN confidence to visualize scoring
ear_range = np.linspace(0.10, 0.40, 30)
cnn_range = np.linspace(0.0, 1.0, 30)
score_grid = np.zeros((len(cnn_range), len(ear_range)))

for i, cnn in enumerate(cnn_range):
    for j, ear in enumerate(ear_range):
        score_grid[i, j] = compute_drowsiness_score(ear=ear, mar=0.4, cnn_confidence=cnn)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.contourf(ear_range, cnn_range, score_grid, levels=20, cmap='RdYlGn_r')
plt.colorbar(im, label='Drowsiness Score')
ax.set_xlabel('EAR (Eye Aspect Ratio)', fontsize=12)
ax.set_ylabel('CNN Drowsy Confidence', fontsize=12)
ax.set_title('Drowsiness Score Heatmap (EAR vs CNN Confidence)', fontsize=14)
ax.axvline(0.25, color='red', linestyle='--', label='EAR Threshold')
ax.axhline(0.65, color='blue', linestyle='--', label='CNN Threshold')
ax.legend()
plt.tight_layout()
plt.savefig('models/results/drowsiness_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Conclusion & Recommendation

Based on the experiments above, select the optimal combo by balancing:
- **Speed**: Must achieve >= 15 FPS for real-time
- **Accuracy**: Highest val accuracy from training
- **Size**: Smaller models are preferred for edge deployment

### Typical Best Choices:
| Use Case | YOLO | CNN | Reason |
|----------|------|-----|--------|
| **Real-time (GPU)** | YOLO11n | EfficientNet-B0 | Best speed-accuracy tradeoff |
| **Edge/Mobile** | YOLOv8n | MobileNetV2 | Smallest + fastest |
| **Max Accuracy** | YOLO11n | ResNet50 | Highest accuracy, acceptable speed |